In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import os
import gc

# ─────────────────────────────────────────────
# Correct dataset path
# ─────────────────────────────────────────────

MASTER = "/kaggle/input/datasets/ayastudentkhoumari/fe-trade/algeria_trade_master_fe.parquet"

OUT = "/kaggle/working/algeria_trade_master_cleaned.parquet"

print("Input exists :", os.path.exists(MASTER))
print("Output path  :", OUT)

Input exists : True
Output path  : /kaggle/working/algeria_trade_master_cleaned.parquet


# Display Dataset Structure + Sample Rows

In [2]:
pf = pq.ParquetFile(MASTER)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Total rows       : {pf.metadata.num_rows:,}")
print(f"Total columns    : {pf.metadata.num_columns}")
print(f"Row groups       : {pf.num_row_groups}")

print("\nCOLUMNS:")
for i, col in enumerate(pf.schema.names, 1):
    print(f"{i:2}. {col}")

# ─────────────────────────────────────────────
# Load ONLY small sample
# ─────────────────────────────────────────────

sample = pf.read_row_group(0).to_pandas().head(5)

print("\n" + "=" * 60)
print("SAMPLE ROWS")
print("=" * 60)

display(sample)

del sample
gc.collect()

DATASET OVERVIEW
Total rows       : 146,823,408
Total columns    : 30
Row groups       : 149

COLUMNS:
 1. year
 2. product_code
 3. exporter
 4. partner
 5. fobvalue
 6. qty
 7. product_description
 8. gdp
 9. gdp_growth
10. population
11. trade_percent_gdp
12. inflation
13. dist
14. contig
15. comlang_off
16. colony
17. continent_j
18. landlocked_j
19. global_imports
20. world_imports
21. global_demand_index
22. hhi
23. algeria_exports
24. market_share
25. penetration_ratio
26. trade_balance
27. product_diversity
28. opportunity_score
29. opportunity_level
30. export_growth

SAMPLE ROWS


,year,product_code,exporter,partner,fobvalue,qty,product_description,gdp,gdp_growth,population,...,global_demand_index,hhi,algeria_exports,market_share,penetration_ratio,trade_balance,product_diversity,opportunity_score,opportunity_level,export_growth
0,2010,610690,AFG,ALB,0.192,0.003,"Blouses, shirts and shirt-blouses: women's or ...",1.208655e+10,2.973155,2913021.0,...,0.000372,0.076895,0.000000,4.290042e-07,0.000376,0.192,7,0.400216,Medium,0.0
1,2010,621590,AFG,ALB,0.845,0.005,"Ties, bow ties and cravats: of textile materia...",1.208655e+10,2.973155,2913021.0,...,0.000044,0.175703,0.000000,1.599126e-05,0.002474,0.845,7,0.399799,Medium,0.0
2,2010,630900,AFG,ALB,0.317,0.017,"Clothing: worn, and other worn articles",1.208655e+10,2.973155,2913021.0,...,0.002385,0.080274,20.184000,3.589011e-05,0.000029,0.317,7,0.400863,Medium,0.0
3,2010,851999,AFG,ALB,0.150,0.001,Sound reproducing apparatus: other than casset...,1.208655e+10,2.973155,2913021.0,...,0.005072,0.308246,474.150024,6.395827e-06,0.000473,0.150,7,0.401610,Medium,0.0
4,2010,852810,AFG,ALB,0.188,0.014,"Television receivers: colour, whether or not c...",1.208655e+10,2.973155,2913021.0,...,0.084941,0.127898,130.994995,5.290362e-06,0.000006,0.188,7,0.425641,High,0.0


0

# Extract Years Safely

In [3]:
meta = pq.read_table(
    MASTER,
    columns=["year"]
).to_pandas()

YEARS = sorted(meta["year"].unique().tolist())

del meta
gc.collect()

print("Years found:")
print(YEARS)
print(f"\nTotal years: {len(YEARS)}")

Years found:
[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2020, 2021, 2022, 2023, 2024]

Total years: 14


# Missing Values Analysis

In [4]:
missing_summary = []

for year in YEARS:

    print(f"Analyzing {year}...")

    yr = pq.read_table(
        MASTER,
        filters=[("year", "=", year)]
    ).to_pandas()

    miss = (
        yr.isna()
        .mean()
        .mul(100)
        .round(2)
        .reset_index()
    )

    miss.columns = ["column", "missing_pct"]
    miss["year"] = year

    missing_summary.append(miss)

    del yr, miss
    gc.collect()

missing_df = pd.concat(missing_summary, ignore_index=True)

print("\nTop missing columns:")
display(
    missing_df
    .sort_values("missing_pct", ascending=False)
    .head(30)
)

Analyzing 2010...
Analyzing 2011...
Analyzing 2012...
Analyzing 2013...
Analyzing 2014...
Analyzing 2015...
Analyzing 2016...
Analyzing 2017...
Analyzing 2018...
Analyzing 2020...
Analyzing 2021...
Analyzing 2022...
Analyzing 2023...
Analyzing 2024...

Top missing columns:


,column,missing_pct,year
400,trade_percent_gdp,9.34,2024
253,contig,7.61,2018
283,contig,7.61,2020
284,comlang_off,7.61,2020
282,dist,7.61,2020
252,dist,7.61,2018
285,colony,7.61,2020
254,comlang_off,7.61,2018
255,colony,7.61,2018
374,comlang_off,7.60,2023


# Main Cleaning Pipeline

In [5]:
writer = None

for year in YEARS:

    print(f"\nCleaning year {year}...", end=" ")

    yr = pq.read_table(
        MASTER,
        filters=[("year", "=", year)]
    ).to_pandas()

    # ────────────────────────────────────────
    # MEMORY OPTIMIZATION
    # ────────────────────────────────────────

    for col in yr.select_dtypes("float64").columns:
        yr[col] = yr[col].astype("float32")

    # ────────────────────────────────────────
    # REMOVE DUPLICATES
    # ────────────────────────────────────────

    yr = yr.drop_duplicates()

    # ────────────────────────────────────────
    # HANDLE INFINITE VALUES
    # ────────────────────────────────────────

    yr = yr.replace([np.inf, -np.inf], np.nan)

    # ────────────────────────────────────────
    # ECONOMIC INDICATORS
    # ────────────────────────────────────────

    econ_cols = [
        "gdp",
        "gdp_growth",
        "population",
        "trade_percent_gdp",
        "inflation"
    ]

    for col in econ_cols:

        if col in yr.columns:

            # country median
            med = (
                yr.groupby("partner")[col]
                .transform("median")
            )

            yr[col] = yr[col].fillna(med)

            # global fallback
            yr[col] = yr[col].fillna(
                yr[col].median()
            )

    # ────────────────────────────────────────
    # TRADE VARIABLES
    # ────────────────────────────────────────

    trade_cols = [
        "qty",
        "netWgt",
        "global_imports",
        "world_imports"
    ]

    for col in trade_cols:

        if col in yr.columns:

            med = (
                yr.groupby("product_code")[col]
                .transform("median")
            )

            yr[col] = yr[col].fillna(med)

            yr[col] = yr[col].fillna(
                yr[col].median()
            )

    # ────────────────────────────────────────
    # ENGINEERED FEATURES
    # ────────────────────────────────────────

    zero_cols = [
        "market_share",
        "penetration_ratio",
        "export_growth",
        "trade_balance",
        "global_demand_index"
    ]

    for col in zero_cols:

        if col in yr.columns:

            yr[col] = yr[col].fillna(0)

    # ────────────────────────────────────────
    # CATEGORICAL VARIABLES
    # ────────────────────────────────────────

    cat_cols = yr.select_dtypes(include="object").columns

    for col in cat_cols:

        yr[col] = yr[col].fillna("Unknown")

    # ────────────────────────────────────────
    # FINAL NUMERIC SAFETY
    # ────────────────────────────────────────

    num_cols = yr.select_dtypes(include=np.number).columns

    for col in num_cols:

        yr[col] = yr[col].fillna(
            yr[col].median()
        )

    # ────────────────────────────────────────
    # WRITE YEAR
    # ────────────────────────────────────────

    table = pa.Table.from_pandas(
        yr,
        preserve_index=False
    )

    if writer is None:

        writer = pq.ParquetWriter(
            OUT,
            table.schema,
            compression="snappy"
        )

    try:
        writer.write_table(table)

    except Exception:
        writer.write_table(table.cast(writer.schema))

    print(f"{len(yr):,} rows")

    del yr, table
    gc.collect()

writer.close()

print("\nDONE")
print(f"Saved file → {OUT}")
print(f"Size       → {os.path.getsize(OUT)/1e6:.1f} MB")


Cleaning year 2010... 9,470,765 rows

Cleaning year 2011... 9,640,712 rows

Cleaning year 2012... 9,937,050 rows

Cleaning year 2013... 10,098,942 rows

Cleaning year 2014... 10,145,672 rows

Cleaning year 2015... 10,499,686 rows

Cleaning year 2016... 10,569,571 rows

Cleaning year 2017... 10,821,738 rows

Cleaning year 2018... 10,924,964 rows

Cleaning year 2020... 10,638,241 rows

Cleaning year 2021... 11,146,476 rows

Cleaning year 2022... 11,154,874 rows

Cleaning year 2023... 11,194,668 rows

Cleaning year 2024... 10,580,049 rows

DONE
Saved file → /kaggle/working/algeria_trade_master_cleaned.parquet
Size       → 5704.3 MB


# Verification

In [6]:
pf = pq.ParquetFile(OUT)

print("=" * 60)
print("CLEANED DATASET")
print("=" * 60)

print(f"Rows       : {pf.metadata.num_rows:,}")
print(f"Columns    : {pf.metadata.num_columns}")
print(f"Row groups : {pf.num_row_groups}")

sample = pf.read_row_group(0).to_pandas().head(5)

print("\nSample:")
display(sample)

del sample
gc.collect()

CLEANED DATASET
Rows       : 146,823,408
Columns    : 30
Row groups : 149

Sample:


,year,product_code,exporter,partner,fobvalue,qty,product_description,gdp,gdp_growth,population,...,global_demand_index,hhi,algeria_exports,market_share,penetration_ratio,trade_balance,product_diversity,opportunity_score,opportunity_level,export_growth
0,2010,610690,AFG,ALB,0.192,0.003,"Blouses, shirts and shirt-blouses: women's or ...",1.208655e+10,2.973155,2913021.0,...,0.000372,0.076895,0.000000,4.290042e-07,0.000376,0.192,7,0.400216,Medium,0.0
1,2010,621590,AFG,ALB,0.845,0.005,"Ties, bow ties and cravats: of textile materia...",1.208655e+10,2.973155,2913021.0,...,0.000044,0.175703,0.000000,1.599126e-05,0.002474,0.845,7,0.399799,Medium,0.0
2,2010,630900,AFG,ALB,0.317,0.017,"Clothing: worn, and other worn articles",1.208655e+10,2.973155,2913021.0,...,0.002385,0.080274,20.184000,3.589011e-05,0.000029,0.317,7,0.400863,Medium,0.0
3,2010,851999,AFG,ALB,0.150,0.001,Sound reproducing apparatus: other than casset...,1.208655e+10,2.973155,2913021.0,...,0.005072,0.308246,474.150024,6.395827e-06,0.000473,0.150,7,0.401610,Medium,0.0
4,2010,852810,AFG,ALB,0.188,0.014,"Television receivers: colour, whether or not c...",1.208655e+10,2.973155,2913021.0,...,0.084941,0.127898,130.994995,5.290362e-06,0.000006,0.188,7,0.425641,High,0.0


2

# Cleaning Statistics

In [7]:
# ─────────────────────────────────────────────
# Compare original vs cleaned dataset
# ─────────────────────────────────────────────

original_pf = pq.ParquetFile(MASTER)
clean_pf    = pq.ParquetFile(OUT)

original_rows = original_pf.metadata.num_rows
clean_rows    = clean_pf.metadata.num_rows

removed_rows = original_rows - clean_rows

print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(f"Original rows : {original_rows:,}")
print(f"Cleaned rows  : {clean_rows:,}")
print(f"Removed rows  : {removed_rows:,}")

print("\nPercentage removed:")
print(f"{(removed_rows / original_rows) * 100:.4f}%")

CLEANING SUMMARY
Original rows : 146,823,408
Cleaned rows  : 146,823,408
Removed rows  : 0

Percentage removed:
0.0000%


In [8]:
import pyarrow.parquet as pq
import pandas as pd
import gc

pf = pq.ParquetFile(OUT)

total_duplicates = 0
total_rows = 0

for i in range(pf.num_row_groups):

    print(f"Checking row group {i+1}/{pf.num_row_groups}...")

    chunk = pf.read_row_group(i).to_pandas()

    dup_count = chunk.duplicated().sum()

    total_duplicates += dup_count
    total_rows += len(chunk)

    print(f"  Rows       : {len(chunk):,}")
    print(f"  Duplicates : {dup_count:,}")

    del chunk
    gc.collect()

print("\n" + "="*60)
print("DUPLICATE SUMMARY")
print("="*60)

print(f"Total rows checked : {total_rows:,}")
print(f"Total duplicates   : {total_duplicates:,}")
print(f"Duplicate %        : {(total_duplicates/total_rows)*100:.6f}%")

Checking row group 1/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 2/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 3/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 4/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 5/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 6/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 7/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 8/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 9/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 10/149...
  Rows       : 33,581
  Duplicates : 0
Checking row group 11/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 12/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 13/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group 14/149...
  Rows       : 1,048,576
  Duplicates : 0
Checking row group

In [9]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

# ─────────────────────────────────────────────
# Open parquet files
# ─────────────────────────────────────────────

orig_pf  = pq.ParquetFile(MASTER)
clean_pf = pq.ParquetFile(OUT)

original_missing = 0
clean_missing    = 0

# ─────────────────────────────────────────────
# Process row group by row group
# ─────────────────────────────────────────────

for i in range(orig_pf.num_row_groups):

    print(f"Checking row group {i+1}/{orig_pf.num_row_groups}...")

    # original chunk
    orig_chunk = orig_pf.read_row_group(i).to_pandas()

    # cleaned chunk
    clean_chunk = clean_pf.read_row_group(i).to_pandas()

    # count missing cells
    orig_na  = orig_chunk.isna().sum().sum()
    clean_na = clean_chunk.isna().sum().sum()

    original_missing += orig_na
    clean_missing    += clean_na

    print(f"  Original missing : {orig_na:,}")
    print(f"  Cleaned missing  : {clean_na:,}")

    del orig_chunk, clean_chunk
    gc.collect()

# ─────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────

filled_cells = original_missing - clean_missing

print("\n" + "="*60)
print("MISSING VALUE CLEANING SUMMARY")
print("="*60)

print(f"Original missing cells : {original_missing:,}")
print(f"Remaining missing cells: {clean_missing:,}")
print(f"Filled missing cells   : {filled_cells:,}")

if original_missing > 0:

    pct = (filled_cells / original_missing) * 100

    print(f"\nPercentage fixed       : {pct:.4f}%")

Checking row group 1/149...
  Original missing : 320,408
  Cleaned missing  : 41,431
Checking row group 2/149...
  Original missing : 300,872
  Cleaned missing  : 35,867
Checking row group 3/149...
  Original missing : 296,095
  Cleaned missing  : 35,982
Checking row group 4/149...
  Original missing : 338,040
  Cleaned missing  : 44,713
Checking row group 5/149...
  Original missing : 849,240
  Cleaned missing  : 33,003
Checking row group 6/149...
  Original missing : 490,077
  Cleaned missing  : 33,044
Checking row group 7/149...
  Original missing : 734,239
  Cleaned missing  : 39,683
Checking row group 8/149...
  Original missing : 242,379
  Cleaned missing  : 31,028
Checking row group 9/149...
  Original missing : 282,147
  Cleaned missing  : 36,987
Checking row group 10/149...
  Original missing : 14,161
  Cleaned missing  : 2,017
Checking row group 11/149...
  Original missing : 325,079
  Cleaned missing  : 43,019
Checking row group 12/149...
  Original missing : 298,179
  Clean

In [10]:
import pyarrow.parquet as pq
import pandas as pd
import gc

pf = pq.ParquetFile(OUT)

missing_by_col = {}

for i in range(pf.num_row_groups):

    print(f"Analyzing row group {i+1}/{pf.num_row_groups}...")

    chunk = pf.read_row_group(i).to_pandas()

    miss = chunk.isna().sum()

    for col, val in miss.items():

        missing_by_col[col] = (
            missing_by_col.get(col, 0) + val
        )

    del chunk
    gc.collect()

missing_df = pd.DataFrame({
    "column": missing_by_col.keys(),
    "missing_cells": missing_by_col.values()
})

missing_df["missing_pct"] = (
    missing_df["missing_cells"]
    /
    pf.metadata.num_rows
) * 100

missing_df = missing_df.sort_values(
    "missing_cells",
    ascending=False
)

print("\nColumns with remaining missing values:\n")

display(missing_df)

Analyzing row group 1/149...
Analyzing row group 2/149...
Analyzing row group 3/149...
Analyzing row group 4/149...
Analyzing row group 5/149...
Analyzing row group 6/149...
Analyzing row group 7/149...
Analyzing row group 8/149...
Analyzing row group 9/149...
Analyzing row group 10/149...
Analyzing row group 11/149...
Analyzing row group 12/149...
Analyzing row group 13/149...
Analyzing row group 14/149...
Analyzing row group 15/149...
Analyzing row group 16/149...
Analyzing row group 17/149...
Analyzing row group 18/149...
Analyzing row group 19/149...
Analyzing row group 20/149...
Analyzing row group 21/149...
Analyzing row group 22/149...
Analyzing row group 23/149...
Analyzing row group 24/149...
Analyzing row group 25/149...
Analyzing row group 26/149...
Analyzing row group 27/149...
Analyzing row group 28/149...
Analyzing row group 29/149...
Analyzing row group 30/149...
Analyzing row group 31/149...
Analyzing row group 32/149...
Analyzing row group 33/149...
Analyzing row group

,column,missing_cells,missing_pct
16,continent_j,5830353,3.970997
0,year,0,0.000000
2,exporter,0,0.000000
3,partner,0,0.000000
4,fobvalue,0,0.000000
1,product_code,0,0.000000
6,product_description,0,0.000000
7,gdp,0,0.000000
8,gdp_growth,0,0.000000
9,population,0,0.000000


In [11]:
# ============================================================
# FINAL ML-READY CLEANING CELL (FIXED VERSION)
# -> handles remaining missing values
# -> fixes categorical errors
# -> guarantees 0 missing values
# -> creates FINAL ML-ready dataset
# ============================================================

import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np
import os
import gc

# ------------------------------------------------------------
# INPUT / OUTPUT
# ------------------------------------------------------------

INPUT_FILE = OUT

FINAL_DATASET = "/kaggle/working/algeria_trade_master_ml_ready.parquet"

pf = pq.ParquetFile(INPUT_FILE)

writer = None

# ============================================================
# STEP 1 — BUILD continent lookup
# ============================================================

print("=" * 60)
print("STEP 1 — Building continent lookup")
print("=" * 60)

continent_lookup = {}

for i in range(pf.num_row_groups):

    chunk = pf.read_row_group(
        i,
        columns=["partner", "continent_j"]
    ).to_pandas()

    # convert safely
    chunk["continent_j"] = chunk["continent_j"].astype("object")

    valid = chunk.dropna(subset=["continent_j"])

    grouped = (
        valid.groupby("partner")["continent_j"]
        .agg(
            lambda x:
            x.mode().iloc[0]
            if not x.mode().empty
            else "Unknown"
        )
    )

    for k, v in grouped.items():

        if k not in continent_lookup:
            continent_lookup[k] = v

    del chunk, valid, grouped
    gc.collect()

print(f"Lookup size: {len(continent_lookup):,}")

# ============================================================
# STEP 2 — FINAL CLEANING
# ============================================================

print("\n" + "=" * 60)
print("STEP 2 — Final cleaning")
print("=" * 60)

for i in range(pf.num_row_groups):

    print(f"Processing row group {i+1}/{pf.num_row_groups}...")

    chunk = pf.read_row_group(i).to_pandas()

    # --------------------------------------------------------
    # FIX continent_j
    # --------------------------------------------------------

    chunk["continent_j"] = (
        chunk["continent_j"]
        .astype("object")
    )

    chunk["continent_j"] = chunk["continent_j"].fillna(
        chunk["partner"].map(continent_lookup)
    )

    chunk["continent_j"] = chunk["continent_j"].fillna("Unknown")

    # --------------------------------------------------------
    # HANDLE categorical/object columns
    # --------------------------------------------------------

    cat_cols = chunk.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in cat_cols:

        chunk[col] = chunk[col].astype("object")

        chunk[col] = chunk[col].fillna("Unknown")

    # --------------------------------------------------------
    # HANDLE numeric columns
    # --------------------------------------------------------

    num_cols = chunk.select_dtypes(
        include=np.number
    ).columns

    for col in num_cols:

        chunk[col] = chunk[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

        chunk[col] = chunk[col].fillna(
            chunk[col].median()
        )

    # --------------------------------------------------------
    # MEMORY OPTIMIZATION
    # --------------------------------------------------------

    for col in chunk.select_dtypes("float64").columns:

        chunk[col] = chunk[col].astype("float32")

    # --------------------------------------------------------
    # WRITE parquet incrementally
    # --------------------------------------------------------

    table = pa.Table.from_pandas(
        chunk,
        preserve_index=False
    )

    if writer is None:

        writer = pq.ParquetWriter(
            FINAL_DATASET,
            table.schema,
            compression="snappy"
        )

    try:

        writer.write_table(table)

    except Exception:

        writer.write_table(
            table.cast(writer.schema)
        )

    del chunk, table
    gc.collect()

writer.close()

# ============================================================
# STEP 3 — VERIFY FINAL DATASET
# ============================================================

print("\n" + "=" * 60)
print("STEP 3 — Verification")
print("=" * 60)

final_pf = pq.ParquetFile(FINAL_DATASET)

missing_total = 0

for i in range(final_pf.num_row_groups):

    chunk = final_pf.read_row_group(i).to_pandas()

    missing_total += chunk.isna().sum().sum()

    del chunk
    gc.collect()

print(f"\nFinal rows       : {final_pf.metadata.num_rows:,}")
print(f"Final columns    : {final_pf.metadata.num_columns}")
print(f"Final row groups : {final_pf.num_row_groups}")

print(f"\nRemaining missing values : {missing_total:,}")

if missing_total == 0:

    print("\nSUCCESS — Dataset is fully ML-ready.")

else:

    print("\nSome missing values still remain.")

print(f"\nSaved file:")
print(FINAL_DATASET)

print(f"\nFile size:")
print(f"{os.path.getsize(FINAL_DATASET)/1e6:.1f} MB")

STEP 1 — Building continent lookup
Lookup size: 212

STEP 2 — Final cleaning
Processing row group 1/149...
Processing row group 2/149...
Processing row group 3/149...
Processing row group 4/149...
Processing row group 5/149...
Processing row group 6/149...
Processing row group 7/149...
Processing row group 8/149...
Processing row group 9/149...
Processing row group 10/149...
Processing row group 11/149...
Processing row group 12/149...
Processing row group 13/149...
Processing row group 14/149...
Processing row group 15/149...
Processing row group 16/149...
Processing row group 17/149...
Processing row group 18/149...
Processing row group 19/149...
Processing row group 20/149...
Processing row group 21/149...
Processing row group 22/149...
Processing row group 23/149...
Processing row group 24/149...
Processing row group 25/149...
Processing row group 26/149...
Processing row group 27/149...
Processing row group 28/149...
Processing row group 29/149...
Processing row group 30/149...
Pr

In [12]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc

# ORIGINAL DATASET
orig_pf = pq.ParquetFile(MASTER)

# FINAL ML-READY DATASET
clean_pf = pq.ParquetFile(
    "/kaggle/working/algeria_trade_master_ml_ready.parquet"
)

original_missing = 0
clean_missing = 0

for i in range(orig_pf.num_row_groups):

    print(f"Checking row group {i+1}/{orig_pf.num_row_groups}...")

    orig_chunk = orig_pf.read_row_group(i).to_pandas()

    clean_chunk = clean_pf.read_row_group(i).to_pandas()

    orig_na = orig_chunk.isna().sum().sum()

    clean_na = clean_chunk.isna().sum().sum()

    original_missing += orig_na
    clean_missing += clean_na

    print(f"  Original missing : {orig_na:,}")
    print(f"  Cleaned missing  : {clean_na:,}")

    del orig_chunk, clean_chunk
    gc.collect()

filled_cells = original_missing - clean_missing

print("\n" + "="*60)
print("FINAL CLEANING SUMMARY")
print("="*60)

print(f"Original missing cells : {original_missing:,}")
print(f"Remaining missing cells: {clean_missing:,}")
print(f"Filled missing cells   : {filled_cells:,}")

if original_missing > 0:

    pct = (filled_cells / original_missing) * 100

    print(f"\nPercentage fixed       : {pct:.4f}%")

Checking row group 1/149...
  Original missing : 320,408
  Cleaned missing  : 0
Checking row group 2/149...
  Original missing : 300,872
  Cleaned missing  : 0
Checking row group 3/149...
  Original missing : 296,095
  Cleaned missing  : 0
Checking row group 4/149...
  Original missing : 338,040
  Cleaned missing  : 0
Checking row group 5/149...
  Original missing : 849,240
  Cleaned missing  : 0
Checking row group 6/149...
  Original missing : 490,077
  Cleaned missing  : 0
Checking row group 7/149...
  Original missing : 734,239
  Cleaned missing  : 0
Checking row group 8/149...
  Original missing : 242,379
  Cleaned missing  : 0
Checking row group 9/149...
  Original missing : 282,147
  Cleaned missing  : 0
Checking row group 10/149...
  Original missing : 14,161
  Cleaned missing  : 0
Checking row group 11/149...
  Original missing : 325,079
  Cleaned missing  : 0
Checking row group 12/149...
  Original missing : 298,179
  Cleaned missing  : 0
Checking row group 13/149...
  Origina